In [0]:

# Task 2: Data Center IoT Transformation
# Bronze Delta -> Cleaning -> Silver Delta
# Shared process_date from Databricks Job

from datetime import datetime

from pyspark.sql.functions import (
    col,
    upper,
    trim
)

from pipeline_logger import logger, save_logs_to_volume


try:
    logger.info("===== DATA CENTER TRANSFORMATION STARTED =====")

    base = "/Volumes/workspace/iot_db/data"

    # STEP 1: Get shared process date from Job
    # Job parameter:
    # {{tasks.00_prepare_date.values.process_date}}

    dbutils.widgets.text(
        "process_date",
        "",
        "Process Date"
    )

    process_date = dbutils.widgets.get(
        "process_date"
    ).strip()

    if not process_date:
        raise ValueError(
            "process_date is required from prepare_date task"
        )

    # Validate date format
    datetime.strptime(process_date, "%Y-%m-%d")

    print("Process Date:", process_date)
    logger.info(f"Processing date: {process_date}")

    # STEP 2: Read Bronze data
    bronze_path = f"{base}/dc_bronze"

    df = (
        spark.read
        .format("delta")
        .load(bronze_path)
        .filter(col("process_date") == process_date)
    )

    bronze_count = df.count()

    print("Bronze records:", bronze_count)
    logger.info(f"Bronze records: {bronze_count}")

    if bronze_count == 0:
        raise ValueError(
            f"No Bronze records found for {process_date}"
        )

    # STEP 3: Remove exact duplicates
    df = df.dropDuplicates()

    after_duplicates = df.count()

    print(
        "Duplicates removed:",
        bronze_count - after_duplicates
    )

    # STEP 4: Standardize text
    for column in [
        "region",
        "sensor_status",
        "cooling_status",
        "alert_level"
    ]:
        df = df.withColumn(
            column,
            upper(trim(col(column)))
        )

    # STEP 5: Convert data types
    for column in [
        "temperature_c",
        "humidity_percent",
        "power_consumption_kw"
    ]:
        df = df.withColumn(
            column,
            col(column).try_cast("double")
        )

    df = df.withColumn(
        "reading_timestamp",
        col("reading_timestamp").try_cast("timestamp")
    )

    df = df.withColumn(
        "ingestion_timestamp",
        col("ingestion_timestamp").try_cast("timestamp")
    )

    # STEP 6: Remove NULL values
    df = df.filter(
        col("reading_id").isNotNull() &
        col("sensor_id").isNotNull() &
        col("data_center_id").isNotNull() &
        col("temperature_c").isNotNull() &
        col("humidity_percent").isNotNull() &
        col("reading_timestamp").isNotNull()
    )

    # STEP 7: Validate temperature and humidity
    df = df.filter(
        col("temperature_c").between(15, 45) &
        col("humidity_percent").between(0, 100)
    )

    # STEP 8: Validate sensor and cooling status
    df = df.filter(
        col("sensor_status").isin(
            "ONLINE",
            "OFFLINE",
            "FAULTY"
        ) &
        col("cooling_status").isin(
            "ACTIVE",
            "DEGRADED",
            "FAILED"
        )
    )

    # STEP 9: Validate region and alert level
    df = df.filter(
        col("region").isin(
            "NORTH",
            "SOUTH",
            "EAST",
            "WEST"
        ) &
        col("alert_level").isin(
            "NORMAL",
            "WARNING",
            "CRITICAL"
        )
    )

    # STEP 10: Validate Data Center IDs
    df = df.filter(
        col("data_center_id").isin(
            "DC01", "DC02", "DC03", "DC04", "DC05",
            "DC06", "DC07", "DC08", "DC09", "DC10"
        )
    )

    # STEP 11: Count clean records
    silver_count = df.count()

    print("Clean Silver records:", silver_count)

    print(
        "Invalid records removed:",
        after_duplicates - silver_count
    )

    logger.info(
        f"Clean Silver records: {silver_count}"
    )

    if silver_count == 0:
        raise ValueError(
            f"No valid Silver records for {process_date}"
        )

    # STEP 12: Display sample data
    display(df.limit(10))

    # STEP 13: Save Silver Delta
    silver_path = f"{base}/dc_silver"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"process_date = '{process_date}'"
        )
        .partitionBy("process_date")
        .save(silver_path)
    )

    logger.info(
        "Silver transformation completed successfully"
    )

    print("DATA CENTER SILVER TRANSFORMATION SUCCESSFUL")
    print("Process Date:", process_date)
    print("Silver Records:", silver_count)
    print("Silver Path:", silver_path)

except Exception:
    logger.exception("Data Center transformation failed")
    raise

finally:
    save_logs_to_volume(dbutils)


Bronze records: 1515000
Duplicates removed: 15000
Clean Silver records: 1409727
Invalid records removed: 90273


reading_id,sensor_id,data_center_id,region,city,rack_id,reading_timestamp,temperature_c,humidity_percent,cooling_status,power_consumption_kw,sensor_status,alert_level,process_date,ingestion_timestamp
25c1a903-4b20-4926-afc1-0a60b76866fd,DC01-R001,DC01,WEST,Mumbai,R001,2026-10-18T00:00:00.000Z,41.2,43.8,FAILED,8.98,ONLINE,CRITICAL,2026-10-18,2026-10-18T08:55:33.538Z
2aaad773-598d-4983-9de8-470772b22e54,DC02-R001,DC02,WEST,Navi Mumbai,R001,2026-10-18T00:00:00.000Z,22.8,57.9,ACTIVE,9.25,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
0e2c8cec-ef3e-4fa7-a7dd-0d56884c0172,DC03-R001,DC03,SOUTH,Chennai,R001,2026-10-18T00:00:01.000Z,24.6,61.8,ACTIVE,6.26,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
25b9eb61-9aab-4bbd-93a7-3f0ed0015b28,DC04-R001,DC04,SOUTH,Hyderabad,R001,2026-10-18T00:00:02.000Z,23.3,64.2,ACTIVE,9.76,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
0fa05464-21da-4f9a-9b5b-b95b9a54864f,DC05-R001,DC05,SOUTH,Bengaluru,R001,2026-10-18T00:00:03.000Z,25.1,57.5,ACTIVE,9.0,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
7eee2bea-6fbb-4491-a3ae-66f2a1193a50,DC06-R001,DC06,NORTH,Noida,R001,2026-10-18T00:00:04.000Z,23.3,37.0,ACTIVE,8.52,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
ece9cbf0-58c3-4aeb-8f13-e2a760713d21,DC07-R001,DC07,NORTH,Greater Noida,R001,2026-10-18T00:00:05.000Z,24.0,41.1,ACTIVE,7.89,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
25e94c98-2306-456f-ae23-4d046edb5adf,DC08-R001,DC08,WEST,Pune,R001,2026-10-18T00:00:06.000Z,26.5,45.4,ACTIVE,7.43,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
52cf3615-cb7c-4c60-903a-555c5f334921,DC10-R001,DC10,WEST,Ahmedabad,R001,2026-10-18T00:00:07.000Z,20.7,57.5,ACTIVE,4.69,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z
4245738f-6b52-4478-a156-a61a6f0de7dd,DC01-R002,DC01,WEST,Mumbai,R002,2026-10-18T00:00:08.000Z,22.3,63.1,ACTIVE,9.45,ONLINE,NORMAL,2026-10-18,2026-10-18T08:55:33.538Z


DATA CENTER SILVER TRANSFORMATION SUCCESSFUL
Silver path: /Volumes/workspace/iot_db/data/dc_silver
Logs saved to /Volumes/workspace/iot_db/data/logs/2026-10-18_app.log
